# ITMS M8 — a fire and smoke detector for the junction cameras

Trains YOLO11-S (COCO-pretrained) to find **fire** and **smoke** in camera frames, on the public
**D-Fire** dataset (21,527 images, CC0, Gaia / Venâncio et al. 2022), plus Bengaluru traffic frames with
no fire as extra negatives. Orange autos, red buses, brake lights, sodium lamps and dusty haze are what a
fire detector confuses on Indian roads, so the negatives matter as much as the fires.

**How to run**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. **Get D-Fire** (the notebook tries these in order; one is enough):
   - it looks in the Drive folder `itms_fire` for a zip (any name) — the most reliable way: open
     https://github.com/gaia-solutions-on-demand/DFireDataset, download "Training, validation and test
     sets" (or "D-Fire dataset") from its OneDrive link in your browser, and put the zip in
     `My Drive/itms_fire/`;
   - or it downloads the Kaggle copy (a free Kaggle API token in Colab's Secrets, key icon, as
     `KAGGLE_USERNAME` and `KAGGLE_KEY`, makes this work);
   - or it tries the OneDrive link directly.
3. Menu **Runtime → Run all**. Allow Google Drive when asked. About 3 hours; if Colab disconnects,
   reconnect and **Run all** again — it continues.
4. At the end `itms_fire_results.zip` downloads (it is also in Drive, `itms_fire`). Put its contents in
   `itms/data/fire/` and tell Claude.

**What it guards against**
- **Test leakage.** D-Fire's test images, and any training image that is a near-copy of one (256-bit
  image fingerprint within 40 bits: frames from the same video), are kept out of training. A download
  without a split is split by groups of near-copies, so no video straddles train and test.
- **Swapped class ids.** Which label id is fire is checked from the boxes' colours (flames are
  saturated orange, smoke is grey) and against the dataset's own names file when there is one; a
  disagreement stops the run.
- **Broken labels.** Out-of-range boxes are clipped, degenerate ones and unknown class ids dropped,
  unreadable images and images without a label file skipped. Everything is counted in the manifest.
- **Choosing on the test set.** The alarm threshold is chosen on the validation split (D-Fire's plus 300
  Bengaluru frames with no fire), then scored once on the test split.
- **Fixed seeds, a pinned library version, and the exact image lists** are recorded.

In [ ]:
# ---- Settings ----
SIZE = 640
EPOCHS = 30
PATIENCE = 8
BATCH = 32
WORKERS = 2
SEED = 26
BASE_WEIGHTS = "yolo11s.pt"      # Ultralytics' COCO-pretrained YOLO11-S
N_NEG_TRAIN = 1500               # Bengaluru frames with no fire added to training
MAX_SIDE = 1280                  # stored copies are at most this wide
NEAR_DUP_BITS = 40               # fingerprint distance (of 256) at or under which an image is a near-copy
VAL_FRACTION = 0.10              # carved from training when the download has no validation split
FA_TARGET = 0.01                 # the alarm threshold keeps false alarms on Bengaluru frames at or under this
RECALL_BAR = 0.85                # set before the run: share of test images with fire or smoke that must alarm
MIN_IMAGES = 1000                # fewer labelled images than this: not the D-Fire download
NAMES = ["fire", "smoke"]        # the output classes, in this order
WORK = "/content/work_fire"
DRIVE_DIR = "/content/drive/MyDrive/itms_fire"
UVH_DRIVE_DIR = "/content/drive/MyDrive/itms_uvh26_finetune"   # the vehicle notebook's Drive folder
ULTRALYTICS_VERSION = "8.4.163"
KAGGLE_DATASET = "sayedgamal99/smoke-fire-detection-yolo"       # the "ready to use" copy D-Fire links to
DFIRE_ONEDRIVE = "https://1drv.ms/u/c/c0bd25b6b048b01d/EbLgD7bES4FDvUN37Grxn8QBF5gIBBc7YV2qklF08GCiBw"
ZIP_NAME = "itms_fire_results.zip"

In [ ]:
# ---- Setup and checks ----
import os, sys, json, time, shutil, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}",
                "onnx", "onnxruntime", "onnxslim", "kagglehub"], check=True)
import torch
assert torch.cuda.is_available(), ("No GPU. Menu: Runtime -> Change runtime type -> T4 GPU -> Save, "
                                   "then Runtime -> Run all again.")
print("GPU:", torch.cuda.get_device_name(0))
free_gb = shutil.disk_usage("/content").free / 1e9
assert free_gb > 25, f"Only {free_gb:.0f} GB free on this runtime; about 25 GB is needed."
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = DRIVE_DIR
except Exception as e:  # noqa: BLE001 -- runs without Drive, just without resume
    print("Google Drive not mounted:", e, "\nSaving to /content only: a disconnect will lose progress.")
    OUT = "/content/itms_fire"
os.makedirs(OUT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
try:  # optional Kaggle token from Colab's Secrets
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:  # noqa: BLE001
            pass
except Exception:  # noqa: BLE001
    pass
print("outputs ->", OUT)

In [ ]:
# ---- Helpers ----
import base64, glob, hashlib, random, re, zipfile
from collections import Counter, defaultdict
from copy import deepcopy
import numpy as np, cv2, requests

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def dhash16(img):
    """256-bit difference hash (the fingerprint the vehicle notebook uses too)."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    s = cv2.resize(g, (17, 16), interpolation=cv2.INTER_AREA)
    return np.packbits((s[:, 1:] > s[:, :-1]).flatten())

def hamming_min(a, bank):
    """Smallest bit distance from each fingerprint in `a` to any in `bank` (both uint8, 32 bytes each)."""
    if len(a) == 0 or len(bank) == 0:
        return np.full(len(a), 256)
    table = np.array([bin(i).count("1") for i in range(256)], np.uint8)
    out = np.empty(len(a), np.int32)
    for i in range(0, len(a), 256):
        x = a[i:i + 256, None, :] ^ bank[None, :, :]
        out[i:i + 256] = table[x].sum(-1, dtype=np.int32).min(1)
    return out

def near_copy_groups(fps, bits):
    """Groups of images that are near-copies of one another (frames of one video), by union-find over every
    pair within `bits`: so that a split can keep each group on one side."""
    parent = list(range(len(fps)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    table = np.array([bin(i).count("1") for i in range(256)], np.uint8)
    for i in range(0, len(fps), 256):
        d = table[fps[i:i + 256, None, :] ^ fps[None, :, :]].sum(-1, dtype=np.int32)
        for a, b in zip(*np.nonzero(d <= bits)):
            a, b = find(i + int(a)), find(int(b))
            if a != b:
                parent[a] = b
    return [find(i) for i in range(len(fps))]

def split_of(path):
    parts = [p.lower() for p in re.split(r"[\\/]", path)]
    for p in reversed(parts[:-1]):
        if p in ("train", "training"):
            return "train"
        if p in ("val", "valid", "validation"):
            return "val"
        if p in ("test", "testing"):
            return "test"
    return None

def label_path_for(img_path):
    stem, _ = os.path.splitext(img_path)
    cands = [stem + ".txt"]
    parts = re.split(r"([\\/])", stem)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() == "images":
            q = parts[:]
            q[i] = "labels" if parts[i] == "images" else "Labels"
            cands.insert(0, "".join(q) + ".txt")
            break
    for c in cands:
        if os.path.exists(c):
            return c
    return None

def parse_labels(path, W, H):
    """YOLO label lines -> [(class id, x1, y1, x2, y2) in pixels], with what was dropped and why."""
    boxes, st = [], Counter()
    for line in open(path, errors="replace"):
        f = line.split()
        if not f:
            continue
        if len(f) != 5:
            st["malformed line"] += 1
            continue
        try:
            k = int(float(f[0]))
            cx, cy, w, h = (float(v) for v in f[1:])
        except ValueError:
            st["malformed line"] += 1
            continue
        if not all(np.isfinite([cx, cy, w, h])):
            st["non-finite box"] += 1
            continue
        x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
        x2, y2 = (cx + w / 2) * W, (cy + h / 2) * H
        c = (max(0.0, x1), max(0.0, y1), min(float(W), x2), min(float(H), y2))
        if c != (x1, y1, x2, y2):
            st["box clipped to the image"] += 1
        if c[2] - c[0] < 2 or c[3] - c[1] < 2:
            st["box empty, tiny or outside the image"] += 1
            continue
        boxes.append((k, *c))
    return boxes, st

def flame_fraction(img, box):
    """Share of a box's pixels that are flame-coloured (saturated, bright, red-orange-yellow)."""
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return 0.0
    h, s, v = cv2.split(cv2.cvtColor(crop, cv2.COLOR_BGR2HSV))
    flame = ((h <= 35) | (h >= 170)) & (s >= 90) & (v >= 150)
    return float(flame.mean())

FP16_MAX = 65504.0

def bn_report(model):
    """BatchNorm health (the vehicle model had broken statistics; a standard model should not)."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)]
    rv = torch.cat([m.running_var.detach().float().flatten().cpu() for m in bns])
    rm = torch.cat([m.running_mean.detach().float().flatten().cpu() for m in bns])
    bad = ~(torch.isfinite(rv) & torch.isfinite(rm))
    return {"bn_layers": len(bns), "channels": int(rv.numel()), "nonfinite_channels": int(bad.sum()),
            "channels_over_fp16_max": int((rv[~bad] > FP16_MAX).sum()),
            "channels_at_fp16_max": int((rv[~bad] == FP16_MAX).sum())}

def letterbox_rect(img, size, stride=32):
    """The letterbox the ITMS detectors use: fit into `size`, pad only up to a multiple of `stride`."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    ch, cw = nh + (size - nh) % stride, nw + (size - nw) % stride
    canvas = np.full((ch, cw, 3), 114, np.uint8)
    top, left = (ch - nh) // 2, (cw - nw) // 2
    canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas, r, (left, top)

def to_input(img, size):
    x, r, pad = letterbox_rect(img, size)
    return np.ascontiguousarray(x[:, :, ::-1].transpose(2, 0, 1)[None], dtype=np.float32) / 255.0, r, pad

def place(src, dst):
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    return dst

In [ ]:
# ---- Get D-Fire (from Drive, Kaggle, or OneDrive) ----
RAW = f"{WORK}/dfire_raw"

def _unzip(zp, dest):
    tmp = dest + ".part"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zp) as z:
        bad = z.testzip()
        assert bad is None, f"{zp} is damaged ({bad}); download it again"
        z.extractall(tmp)
    os.replace(tmp, dest)

def acquire():
    if os.path.isdir(RAW) and any(os.scandir(RAW)):
        return "already here", RAW
    zips = [p for p in glob.glob(f"{OUT}/*.zip") if os.path.basename(p) != ZIP_NAME]
    if zips:
        _unzip(sorted(zips)[0], RAW)
        return f"Drive zip {os.path.basename(sorted(zips)[0])}", RAW
    errors = []
    try:
        import kagglehub
        path = kagglehub.dataset_download(KAGGLE_DATASET)
        shutil.copytree(path, RAW + ".part", dirs_exist_ok=True)
        os.replace(RAW + ".part", RAW)
        return f"Kaggle {KAGGLE_DATASET}", RAW
    except Exception as e:  # noqa: BLE001
        errors.append(f"Kaggle: {e}")
    try:
        token = base64.urlsafe_b64encode(DFIRE_ONEDRIVE.encode()).decode().rstrip("=")
        url = f"https://api.onedrive.com/v1.0/shares/u!{token}/root/content"
        zp = f"{WORK}/dfire_onedrive.zip"
        with requests.get(url, stream=True, timeout=120, allow_redirects=True) as r:
            r.raise_for_status()
            with open(zp + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.replace(zp + ".part", zp)
        _unzip(zp, RAW)
        return "OneDrive", RAW
    except Exception as e:  # noqa: BLE001
        errors.append(f"OneDrive: {e}")
    raise RuntimeError("Could not get D-Fire automatically:\n  " + "\n  ".join(errors) +
                       f"\nDownload the zip from https://github.com/gaia-solutions-on-demand/DFireDataset in "
                       f"your browser, put it in Drive folder {DRIVE_DIR}, then Run all again.")

DS = f"{WORK}/ds"
MANIFEST = f"{OUT}/fire_manifest.json"
TAR = f"{OUT}/fire_dataset.tar"
NEED_BUILD = not (os.path.exists(MANIFEST) and os.path.exists(TAR))
if NEED_BUILD:
    SOURCE, RAW = acquire()
    print("D-Fire from:", SOURCE)
else:
    print("the prepared dataset is in Drive; nothing to download")

In [ ]:
# ---- Read it: images, labels, splits, and which id is fire ----
def scan(root):
    items, skipped = [], Counter()
    for dirpath, _, files in os.walk(root):
        for fn in sorted(files):
            if not fn.lower().endswith(IMG_EXT) or "__MACOSX" in dirpath:
                continue
            p = os.path.join(dirpath, fn)
            lab = label_path_for(p)
            if lab is None:
                skipped["no label file"] += 1
                continue
            items.append({"path": p, "label": lab, "split": split_of(os.path.relpath(p, root))})
    return items, skipped

def names_file(root):
    """Class names shipped with the download, if any: {id: name}."""
    for p in glob.glob(f"{root}/**/*.yaml", recursive=True) + glob.glob(f"{root}/**/*.yml", recursive=True):
        txt = open(p, errors="replace").read()
        m = re.search(r"names\s*:\s*\[([^\]]*)\]", txt)
        if m:
            vals = [v.strip().strip("'\"").lower() for v in m.group(1).split(",") if v.strip()]
            return {i: v for i, v in enumerate(vals)}, os.path.relpath(p, root)
        m = re.findall(r"^\s*(\d+)\s*:\s*['\"]?([A-Za-z_ -]+?)['\"]?\s*$", txt, flags=re.M)
        if m and "names" in txt:
            return {int(i): v.strip().lower() for i, v in m}, os.path.relpath(p, root)
    for p in glob.glob(f"{root}/**/classes.txt", recursive=True):
        vals = [v.strip().lower() for v in open(p) if v.strip()]
        return {i: v for i, v in enumerate(vals)}, os.path.relpath(p, root)
    return None, None

FIRE_WORDS, SMOKE_WORDS = {"fire", "flame", "flames"}, {"smoke"}

def identify_classes(items):
    """Which label id is fire: the colour of its boxes, checked against the download's names file."""
    rng = random.Random(SEED)
    per_class = defaultdict(list)
    for it in rng.sample(items, min(3000, len(items))):
        img = cv2.imread(it["path"])
        if img is None:
            continue
        H, W = img.shape[:2]
        for k, *b in parse_labels(it["label"], W, H)[0]:
            if len(per_class[k]) < 400:
                per_class[k].append(flame_fraction(img, b))
    flame = {k: float(np.median(v)) for k, v in per_class.items() if len(v) >= 20}
    print("median flame-coloured share of box pixels, by label id:", {k: round(v, 3) for k, v in flame.items()})
    assert len(flame) == 2, f"expected two label ids with boxes, found {sorted(flame)}"
    fire_id, smoke_id = max(flame, key=flame.get), min(flame, key=flame.get)
    assert flame[fire_id] >= 0.05 and flame[fire_id] >= 2 * max(flame[smoke_id], 0.01), (
        "The box colours do not say which label id is fire. Stop and look at a few images with their labels.")
    declared, declared_from = names_file(RAW)
    print("class names file:", declared_from, declared)
    if declared:
        assert declared.get(fire_id, "fire") in FIRE_WORDS and declared.get(smoke_id, "smoke") in SMOKE_WORDS, (
            f"The dataset's names file {declared} disagrees with the box colours (fire looks like id {fire_id}).")
    return {fire_id: NAMES.index("fire"), smoke_id: NAMES.index("smoke")}, flame, declared

if NEED_BUILD:
    items, scan_skips = scan(RAW)
    print(len(items), "labelled images;", dict(scan_skips))
    assert len(items) >= MIN_IMAGES, "too few labelled images: is this the D-Fire download?"
    print("splits in the download:", dict(Counter(it["split"] for it in items)))
    ID_MAP, flame, declared = identify_classes(items)
    print("label ids ->", {k: NAMES[v] for k, v in ID_MAP.items()})

In [ ]:
# ---- Build the training set (or restore it from Drive) ----
SETTINGS = {"SEED": SEED, "N_NEG_TRAIN": N_NEG_TRAIN, "MAX_SIDE": MAX_SIDE, "NEAR_DUP_BITS": NEAR_DUP_BITS,
            "VAL_FRACTION": VAL_FRACTION}

def bengaluru_negatives():
    """Bengaluru traffic frames with no fire, from the vehicle notebook's dataset in Drive: its training frames
    (some go into training here) and its 300 hold-out frames (validation here). The 250 vehicle *test* frames
    are not in it; the laptop uses them for the final false-alarm count."""
    tar = f"{UVH_DRIVE_DIR}/dataset.tar"
    if not os.path.exists(tar):
        print("NOTE: no Bengaluru frames found at", tar, "- training on D-Fire alone; the threshold is then "
              "chosen on D-Fire's own negatives.")
        return [], []
    dest = f"{WORK}/uvh"
    if not os.path.isdir(f"{dest}/ds"):
        os.makedirs(dest, exist_ok=True)
        subprocess.run(["tar", "-xf", tar, "-C", dest], check=True)
    tr = sorted(glob.glob(f"{dest}/ds/images/train/*.jpg"))
    va = sorted(glob.glob(f"{dest}/ds/images/val/*.jpg"))
    random.Random(SEED).shuffle(tr)
    return tr[:N_NEG_TRAIN], va

def write_image(img, dst):
    h, w = img.shape[:2]
    if max(h, w) > MAX_SIDE:
        s = MAX_SIDE / max(h, w)
        img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    assert cv2.imwrite(dst, img, [cv2.IMWRITE_JPEG_QUALITY, 95])

def build():
    for d in ("images/train", "images/val", "images/test", "labels/train", "labels/val", "labels/test"):
        os.makedirs(f"{DS}/{d}", exist_ok=True)
    skipped, box_stats = defaultdict(list), Counter()
    for k, n in scan_skips.items():
        skipped[k].extend([None] * n)       # counted during the scan (no paths kept)
    recs = []   # (item, fingerprint, checksum, boxes)
    for it in items:
        img = cv2.imread(it["path"])
        if img is None:
            skipped["does not open"].append(it["path"])
            continue
        H, W = img.shape[:2]
        boxes, st = parse_labels(it["label"], W, H)
        box_stats.update(st)
        unknown = [b for b in boxes if b[0] not in ID_MAP]
        if unknown:
            box_stats["box with an unknown class id dropped"] += len(unknown)
            boxes = [b for b in boxes if b[0] in ID_MAP]
        recs.append((it, dhash16(img), hashlib.sha256(open(it["path"], "rb").read()).hexdigest(), boxes))
    # splits: the download's own; validation carved from training when it has none (seeded)
    have = {r[0]["split"] for r in recs}
    split_note = "the download's train / val / test split"
    if "test" not in have:
        # no split in the download: a seeded one that keeps each group of near-copies on one side
        group = near_copy_groups(np.stack([r[1] for r in recs]), NEAR_DUP_BITS)
        members = defaultdict(list)
        for i, gid in enumerate(group):
            members[gid].append(i)
        gids = sorted(members)
        biggest = max(len(v) for v in members.values())
        if biggest > 0.1 * len(recs):
            print(f"WARNING: one group of near-copies holds {biggest} of {len(recs)} images; the split is lumpy")
        random.Random(SEED).shuffle(gids)
        targets = [("test", 0.2 * len(recs)), ("val", VAL_FRACTION * len(recs))]
        phase, filled = 0, Counter()
        for gid in gids:           # test fills first, then validation (each gets at least one group), then train
            if phase < 2 and filled[targets[phase][0]] >= targets[phase][1]:
                phase += 1
            sp = targets[phase][0] if phase < 2 else "train"
            for i in members[gid]:
                recs[i][0]["split"] = sp
            filled[sp] += len(members[gid])
        have = {r[0]["split"] for r in recs}
        split_note = (f"no split in the download: a seeded one by groups of near-copies ({len(gids)} groups), "
                      f"test 20 %, validation {VAL_FRACTION:.0%}")
    if "val" not in have:
        tr = [i for i, r in enumerate(recs) if r[0]["split"] == "train"]
        random.Random(SEED).shuffle(tr)
        for i in tr[:int(len(tr) * VAL_FRACTION)]:
            recs[i][0]["split"] = "val"
        split_note = f"the download's train / test split; validation = a seeded {VAL_FRACTION:.0%} of train"
    orphans = [r for r in recs if r[0]["split"] is None]
    for r in orphans:
        skipped["not in a train / val / test folder"].append(r[0]["path"])
    recs = [r for r in recs if r[0]["split"] is not None]
    # keep the test images, and near-copies of them, out of training and validation
    test = [r for r in recs if r[0]["split"] == "test"]
    test_sha = {r[2] for r in test}
    bank = np.stack([r[1] for r in test])
    others = [r for r in recs if r[0]["split"] != "test"]
    dist = hamming_min(np.stack([r[1] for r in others]), bank)
    kept = list(test)
    for r, d in zip(others, dist):
        if r[2] in test_sha:
            skipped[f"{r[0]['split']}: identical to a test image"].append(r[0]["path"])
        elif d <= NEAR_DUP_BITS:
            skipped[f"{r[0]['split']}: near-copy of a test image"].append(r[0]["path"])
        else:
            kept.append(r)
    # training images that are near-copies of validation images are counted, not dropped: validation only
    # picks the best epoch, and the alarm threshold is set by false alarms on Bengaluru frames, not by recall
    val = [r for r in kept if r[0]["split"] == "val"]
    train = [r for r in kept if r[0]["split"] == "train"]
    near_val = 0
    if val and train:
        near_val = int((hamming_min(np.stack([r[1] for r in train]), np.stack([r[1] for r in val]))
                        <= NEAR_DUP_BITS).sum())
    n_before = sum(1 for r in recs if r[0]["split"] == "train")
    n_after = sum(1 for r in kept if r[0]["split"] == "train")
    assert n_after > 0, "every training image is a near-copy of a test image: the split is unusable"
    if n_after < 0.75 * n_before:
        print(f"WARNING: keeping the test clean removed {n_before - n_after} of {n_before} training images "
              "(frames of the same videos on both sides of the download's split)")
    names_out, counts = [], {sp: Counter() for sp in ("train", "val", "test")}
    for n, (it, _, sha, boxes) in enumerate(kept):
        sp = it["split"]
        stem = f"df{n:05d}"
        img = cv2.imread(it["path"])
        H, W = img.shape[:2]
        write_image(img, f"{DS}/images/{sp}/{stem}.jpg")
        with open(f"{DS}/labels/{sp}/{stem}.txt", "w") as f:
            for k, x1, y1, x2, y2 in boxes:
                c = ID_MAP[k]
                f.write(f"{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} "
                        f"{(y2 - y1) / H:.6f}\n")
                counts[sp][NAMES[c]] += 1
        counts[sp]["images"] += 1
        counts[sp]["images with fire or smoke" if boxes else "images with neither"] += 1
        names_out.append({"stem": stem, "split": sp, "source": os.path.relpath(it["path"], RAW), "sha256": sha})
    neg_tr, neg_va = bengaluru_negatives()
    for sp, files in (("train", neg_tr), ("val", neg_va)):
        for p in files:
            stem = "blr_" + os.path.splitext(os.path.basename(p))[0]
            shutil.copy(p, f"{DS}/images/{sp}/{stem}.jpg")
            open(f"{DS}/labels/{sp}/{stem}.txt", "w").close()
            counts[sp]["Bengaluru frames with no fire"] += 1
            names_out.append({"stem": stem, "split": sp, "source": "uvh26:" + os.path.basename(p)})
    assert counts["val"]["images with fire or smoke"] > 0, "no validation image has fire or smoke"
    man = {"settings": dict(SETTINGS), "source": SOURCE, "split": split_note, "id_map": {str(k): NAMES[v]
           for k, v in ID_MAP.items()}, "flame_share_by_id": flame, "declared_names": declared,
           "counts": {sp: dict(c) for sp, c in counts.items()}, "box_stats": dict(box_stats),
           "skipped": {k: len(v) for k, v in skipped.items()},
           "train_near_copies_of_validation_kept": near_val,
           "skipped_examples": {k: [p for p in v if p][:20] for k, v in skipped.items()}, "images": names_out}
    json.dump(man, open(MANIFEST, "w"), indent=1)
    subprocess.run(["tar", "-cf", TAR + ".part", "-C", WORK, "ds"], check=True)
    os.replace(TAR + ".part", TAR)
    return man

if os.path.exists(MANIFEST) and os.path.exists(TAR):
    man = json.load(open(MANIFEST))
    assert man["settings"] == SETTINGS, (
        f"The dataset in Drive was made with {man['settings']}, these settings are {SETTINGS}. Put them back, "
        "or move fire_dataset.tar and fire_manifest.json out of the Drive folder to build a new one.")
    if not os.path.isdir(DS):
        print("restoring the prepared dataset from Drive...")
        subprocess.run(["tar", "-xf", TAR, "-C", WORK], check=True)
else:
    man = build()
for sp in ("train", "val", "test"):
    on_disk = len(os.listdir(f"{DS}/images/{sp}"))
    assert on_disk == sum(1 for r in man["images"] if r["split"] == sp) == len(os.listdir(f"{DS}/labels/{sp}"))
print("split:", man["split"])
print("counts:", json.dumps(man["counts"], indent=1))
print("skipped:", man["skipped"])
print("boxes:", man["box_stats"])

In [ ]:
# ---- Look at the labels: fire (orange) and smoke (grey) boxes on eight training images ----
from IPython.display import Image as _Img, display
COL = {0: (0, 128, 255), 1: (200, 200, 200)}

def sheet(paths, boxes_of, out, cols=4, tile=(400, 300)):
    tiles = []
    for p in paths:
        img = cv2.imread(p)
        for k, x1, y1, x2, y2, *rest in boxes_of(p, img):
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), COL[k], 2)
            txt = NAMES[k] + (f" {rest[0]:.2f}" if rest else "")
            cv2.putText(img, txt, (int(x1), max(14, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COL[k], 2)
        tiles.append(cv2.resize(img, tile))
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    grid = np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])
    cv2.imwrite(out, grid, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return out

def label_boxes(p, img):
    H, W = img.shape[:2]
    lab = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    out = []
    for line in open(lab):
        k, cx, cy, w, h = line.split()
        k, cx, cy, w, h = int(k), float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        out.append((k, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
    return out

with_boxes = [f"{DS}/images/train/{r['stem']}.jpg" for r in man["images"] if r["split"] == "train"
              and os.path.getsize(f"{DS}/labels/train/{r['stem']}.txt") > 0]
pick = random.Random(SEED + 1).sample(with_boxes, min(8, len(with_boxes)))
display(_Img(sheet(pick, label_boxes, f"{OUT}/label_check.jpg")))

In [ ]:
# ---- Train (continues a run after a disconnect) ----
from ultralytics import YOLO
from ultralytics.utils.torch_utils import unwrap_model
YAML = f"{WORK}/fire.yaml"
with open(YAML, "w") as f:
    f.write(f"path: {DS}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n")
    f.write("".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
RUNS = f"{OUT}/runs"
RUN = f"{RUNS}/fire_{SIZE}"
REPORT_PATH = f"{OUT}/fire_report.json"
report = json.load(open(REPORT_PATH)) if os.path.exists(REPORT_PATH) else {}
report.update({"ultralytics": ULTRALYTICS_VERSION, "torch": torch.__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED,
               "settings": {"SIZE": SIZE, "EPOCHS": EPOCHS, "PATIENCE": PATIENCE, "BATCH": BATCH,
                            "BASE_WEIGHTS": BASE_WEIGHTS, "FA_TARGET": FA_TARGET, "RECALL_BAR": RECALL_BAR}})

def save_report():
    json.dump(report, open(REPORT_PATH + ".part", "w"), indent=1)
    os.replace(REPORT_PATH + ".part", REPORT_PATH)

BAD = {"epoch": -1, "n": 0, "bad": 0}

def loss_guard(trainer):
    """Mixed precision may skip the odd overflowing batch; more than 5 % of an epoch's batches is a broken run."""
    if trainer.epoch != BAD["epoch"]:
        BAD.update(epoch=trainer.epoch, n=0, bad=0)
    BAD["n"] += 1
    if trainer.loss is not None and not torch.isfinite(trainer.loss).all():
        BAD["bad"] += 1
    if BAD["n"] >= 50 and BAD["bad"] > 0.05 * BAD["n"]:
        raise RuntimeError(f"{BAD['bad']} of {BAD['n']} batches in epoch {trainer.epoch + 1} gave a non-finite "
                           "loss. Stop here and send Claude this output.")

def check_start(trainer):
    rep = bn_report(unwrap_model(trainer.model))
    assert rep["nonfinite_channels"] == 0 and rep["channels_at_fp16_max"] == 0, rep

last = f"{RUN}/weights/last.pt"
t0 = time.time()
if os.path.exists(last) and torch.load(last, map_location="cpu", weights_only=False).get("optimizer") is None:
    print("training already finished")
else:
    if os.path.exists(last):
        yolo = YOLO(last)
        yolo.add_callback("on_train_batch_end", loss_guard)
        yolo.train(resume=True)
    else:
        yolo = YOLO(BASE_WEIGHTS)
        yolo.add_callback("on_train_batch_end", loss_guard)
        yolo.add_callback("on_pretrain_routine_end", check_start)
        yolo.train(data=YAML, imgsz=SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, close_mosaic=5,
                   cos_lr=True, seed=SEED, deterministic=True, workers=WORKERS, cache=False, amp=True,
                   project=RUNS, name=f"fire_{SIZE}", exist_ok=True, plots=True, save_period=-1)
    report.setdefault("sessions", []).append({"minutes": round((time.time() - t0) / 60),
                                              "batch_used": int(yolo.trainer.batch_size)})
    save_report()

In [ ]:
# ---- The model in full precision, its alarm threshold, and the test ----
PT = f"{OUT}/itms_fire_{SIZE}.pt"
ONNX = f"{OUT}/itms_fire_{SIZE}.onnx"
best = YOLO(f"{RUN}/weights/best.pt")
assert [best.names[i] for i in sorted(best.names)] == NAMES, best.names
bn = bn_report(best.model)
report["bn"] = bn
assert bn["nonfinite_channels"] == 0 and bn["channels_at_fp16_max"] == 0, (
    "the saved model has clipped BatchNorm statistics (the vehicle model's problem); tell Claude", bn)
if not os.path.exists(PT):
    ck = {"epoch": -1, "best_fitness": None, "model": deepcopy(best.model).float().cpu(), "ema": None,
          "optimizer": None, "train_args": dict(best.ckpt.get("train_args", {}))}
    torch.save(ck, PT + ".part")
    os.replace(PT + ".part", PT)
net = YOLO(PT).model.float().eval()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net.to(DEV)

def scores(paths):
    """Each image's highest fire and smoke score, from the raw head (no NMS needed for 'is there any')."""
    out = np.zeros((len(paths), len(NAMES)), np.float32)
    with torch.no_grad():
        for i, p in enumerate(paths):
            x, _, _ = to_input(cv2.imread(p), SIZE)
            y = net(torch.from_numpy(x).to(DEV))
            y = (y[0] if isinstance(y, (list, tuple)) else y)[0]
            out[i] = y[4:4 + len(NAMES)].amax(1).float().cpu().numpy()
    return out

def split_table(sp):
    rows = [r for r in man["images"] if r["split"] == sp]
    paths = [f"{DS}/images/{sp}/{r['stem']}.jpg" for r in rows]
    has = np.zeros((len(rows), len(NAMES)), bool)
    for i, r in enumerate(rows):
        for line in open(f"{DS}/labels/{sp}/{r['stem']}.txt"):
            has[i, int(line.split()[0])] = True
    blr = np.array([r["stem"].startswith("blr_") for r in rows])
    return paths, has, blr

GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
if "threshold" not in report:
    vp, vhas, vblr = split_table("val")
    vs = scores(vp)
    vany, vpos = vs.max(1), vhas.any(1)
    rows = []
    for t in GRID:
        alarm = vany >= t
        rows.append({"t": float(t), "recall": float(alarm[vpos].mean()),
                     "fa_dfire_negatives": float(alarm[~vpos & ~vblr].mean()) if (~vpos & ~vblr).any() else None,
                     "fa_bengaluru": float(alarm[vblr].mean()) if vblr.any() else None})
    key = "fa_bengaluru" if vblr.any() else "fa_dfire_negatives"
    target = FA_TARGET if vblr.any() else 0.05
    ok = [r for r in rows if r[key] is not None and r[key] <= target]
    assert ok, f"no threshold keeps {key} at or under {target}"
    chosen = ok[0]            # the lowest such threshold: the most fires caught within the false-alarm budget
    report["threshold"] = {"t": chosen["t"], "chosen_on": f"validation, {key} <= {target}", "at_t": chosen,
                           "curve": rows}
    save_report()
    # the Bengaluru frames that looked most like fire: a real false alarm, or real smoke nobody labelled?
    order = np.argsort(-vany * vblr)[:12]
    top = [vp[i] for i in order if vblr[i]]
    if top:
        pm = YOLO(PT)

        def pred_boxes(p, img):
            r = pm.predict(img, imgsz=SIZE, conf=0.1, verbose=False)[0]
            return [(int(c), *b, float(sc)) for b, sc, c in
                    zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())]
        sheet(top, pred_boxes, f"{OUT}/bengaluru_negatives_review.jpg")
        display(_Img(f"{OUT}/bengaluru_negatives_review.jpg"))
T = report["threshold"]["t"]
print("alarm threshold (chosen on validation):", report["threshold"]["at_t"])

if "test" not in report:
    tp, thas, tblr = split_table("test")
    ts = scores(tp)
    tany, tpos = ts.max(1), thas.any(1)
    res = {"images": int(len(tp)), "with_fire_or_smoke": int(tpos.sum()),
           "recall_any": float((tany[tpos] >= T).mean()),
           "recall_fire": float((ts[thas[:, 0], 0] >= T).mean()) if thas[:, 0].any() else None,
           "recall_smoke": float((ts[thas[:, 1], 1] >= T).mean()) if thas[:, 1].any() else None,
           "false_alarms_on_negatives": float((tany[~tpos] >= T).mean()) if (~tpos).any() else None}
    r = YOLO(PT).val(data=YAML, imgsz=SIZE, batch=16, split="test", half=False, plots=False, verbose=False)
    res["box_ap50"] = {NAMES[int(c)]: round(float(r.box.ap50[i]), 4) for i, c in enumerate(r.box.ap_class_index)}
    res["box_map50"] = round(float(r.box.map50), 4)
    res["bar"] = {"recall_any_at_least": RECALL_BAR, "passes_in_colab": res["recall_any"] >= RECALL_BAR,
                  "still_to_check_on_the_laptop": "false alarms on the 250 Bengaluru test frames <= 1 %"}
    report["test"] = res
    save_report()
    missed = [tp[i] for i in np.where(tpos & (tany < T))[0][:12]]
    if missed:
        sheet(missed, label_boxes, f"{OUT}/test_misses.jpg")
print("test (scored once):", json.dumps(report["test"], indent=1))

In [ ]:
# ---- Export to ONNX and check it against the full-precision model ----
import onnxruntime as ort
if not (os.path.exists(ONNX) and report.get("onnx_sha256") == sha256_file(ONNX)):
    exp = YOLO(PT).export(format="onnx", dynamic=True, imgsz=SIZE, half=False, simplify=True, device="cpu")
    place(exp, ONNX)
    s = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
    ref = YOLO(PT).model.float().cpu().eval()
    probe = [f"{DS}/images/test/{r['stem']}.jpg" for r in man["images"] if r["split"] == "test"][:4]
    worst = 0.0
    for p in probe:
        x, _, _ = to_input(cv2.imread(p), SIZE)
        a = s.run(None, {s.get_inputs()[0].name: x})[0]
        with torch.no_grad():
            b = ref(torch.from_numpy(x))
        b = (b[0] if isinstance(b, (list, tuple)) else b).numpy()
        assert a.shape == b.shape and a.shape[1] == 4 + len(NAMES), (a.shape, b.shape)
        assert np.isfinite(a).all()
        worst = max(worst, float(np.abs(a[:, 4:] - b[:, 4:]).max()))
    assert worst < 1e-3, ("the exported model disagrees with the original", worst)
    report.update({"onnx_max_score_diff": float(f"{worst:.2e}"), "onnx_sha256": sha256_file(ONNX),
                   "pt_sha256": sha256_file(PT)})
    save_report()
print("export checked:", report["onnx_max_score_diff"])

In [ ]:
# ---- Package and download ----
zp = f"{OUT}/{ZIP_NAME}"
with zipfile.ZipFile(zp + ".part", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ("fire_report.json", "fire_manifest.json", "label_check.jpg", "bengaluru_negatives_review.jpg",
               "test_misses.jpg", os.path.basename(ONNX), os.path.basename(PT)):
        if os.path.exists(f"{OUT}/{fn}"):
            z.write(f"{OUT}/{fn}", fn)
    for fn in ("results.csv", "args.yaml", "results.png", "confusion_matrix.png"):
        if os.path.exists(f"{RUN}/{fn}"):
            z.write(f"{RUN}/{fn}", f"runs/fire_{SIZE}/{fn}")
os.replace(zp + ".part", zp)
print("saved", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
try:
    from google.colab import files as _files
    _files.download(zp)
except Exception:  # noqa: BLE001
    print("Download it from Drive: folder itms_fire")